Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

# Read bronze table

In [0]:
df = spark.table("bikescatalog.bronze.erp_loc_a101")

# Silver Transformations

# Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

# Customer ID Cleanup

In [0]:
df = df.withColumn("CID", F.regexp_replace(col("CID"), "-",""))

# Country Normalization

In [0]:
df = df.withColumn("CNTRY",
        F.when(col("CNTRY") == "DE", "Germany")
        .when(col("CNTRY").isin("US", "USA"), "United States")
        .when((col("CNTRY") == '') | col("CNTRY").isNull(), "n/a")
        .otherwise(col("CNTRY"))
)
    

# Renaming Columns

In [0]:
RENAME_MAP = {
    "CID" : "customer_number",
    "CNTRY" : "country" 
}
for old_name, new_name in RENAME_MAP.items():
   df = df.withColumnRenamed(old_name, new_name)

# Sanity check of dataframe

In [0]:
df.limit(10).display()

# writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("bikescatalog.silver.erp_customer_location")

# Sanity check of silver table

In [0]:
%sql
select * from bikescatalog.silver.erp_customer_location limit 10